# Courier routes with time windows, augmented by Machine Learning

**Machine Learning & Logistics Optimisation project — LaDe-P dataset (Cainiao), city of Yantai**

Professor: Mr Slim Ayedi

---

## A. Introduction

### The business context
In *last-mile* logistics, a courier leaves in the morning and must **collect parcels** from customers (shops, individuals). Each parcel has a **time window promised** to the customer (for example 9 am – 11 am). Requests arrive **throughout the day**: the courier does not know all of their tasks at the start.

### The logistics problem
Every time a pickup is finished, the courier has to decide: **"Where do I go now?"**
A bad visiting order means useless kilometres, lost time and missed time windows.

### The decision to optimise
**The order in which pickup points are visited**, decided at every stop among the tasks already known.
This is a dynamic variant of the **Travelling Salesman Problem with Time Windows (TSPTW)**, a classic combinatorial optimisation problem (NP-hard).

### Why Machine Learning
An optimisation algorithm needs to know **the time required to go from one point to another**. That time is not in the data: it depends on distance, type of place, time of day, neighbourhood… ML **learns this time** from nearly one million real trips, then gives the optimiser a **time matrix** that is more realistic than a plain "distance ÷ average speed" formula.

> **In one sentence:** ML *predicts* travel times; optimisation *decides* the visiting order from these predictions.

### The KPIs tracked
| KPI | Meaning | Unit |
|---|---|---|
| Distance travelled | sum of distances between successive visits (exact calculation, no model) | km |
| Travel time | sum of travel + service times | min |
| Route duration | from the first to the last pickup (includes waiting for new tasks) | min |
| Late pickups | number of visits made after the end of their window | count |
| Minutes late | cumulated lateness | min |

### The three strategies compared
1. **Field reference**: the order *actually* followed by the courier (recorded in the data).
2. **Scenario 1 — Without ML**: the same optimiser, but with a traditional "average service time + distance ÷ average speed" formula.
3. **Scenario 2 — With ML**: the same optimiser, with the time matrix predicted by the ML model.

Comparing 2 and 3 isolates the **added value of ML**; comparing 1 with 2/3 measures the **added value of optimisation**.

## B. Importing and exploring the dataset

### B.1 Libraries and parameters
All libraries are free and already installed in Google Colab: `pandas`, `numpy`, `matplotlib`, `scikit-learn` and `scipy` (for the exact **HiGHS** MILP solver). No additional installation is needed.

In [ ]:
# B.1 — Imports and global parameters

import json, time, math, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy.optimize import milp, LinearConstraint, Bounds
from scipy.stats import wilcoxon

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
SEED = 42                      # random seed: reproducible results
rng = np.random.default_rng(SEED)

#  Project folders (works locally from notebook/ or in Colab) ---
BASE = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
DATA_PATH = BASE / "data" / "pickup_yt.csv"
FIG_DIR = BASE / "rapport" / "figures_en"
EXP_DIR = BASE / "exports_en"
WEB_DIR = BASE / "web" / "data_en"
for d in (FIG_DIR, EXP_DIR, WEB_DIR):
    d.mkdir(parents=True, exist_ok=True)

#  Shared plot style (validated palette, print-friendly) ---
C_REEL, C_SANS, C_AVEC, C_GRIS = "#8a8985", "#eb6834", "#2a78d6", "#52514e"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "font.size": 10,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25,
    "axes.edgecolor": "#888", "legend.frameon": False,
})

def sauver(fig, nom):
    """Save a figure as a high-quality PNG for the report."""
    fig.savefig(FIG_DIR / f"{nom}.png", bbox_inches="tight", facecolor="white")

print("Project folder:", BASE)
print("Data file:", DATA_PATH, "| found:", DATA_PATH.exists())

### B.2 Loading the file
The file `pickup_yt.csv` comes from the public **LaDe** dataset (Cainiao, Apache-2.0 licence), *pickup* subset, city of **Yantai**.
Download: https://huggingface.co/datasets/Cainiao-AI/LaDe/tree/main (`pickup` folder).

> In Google Colab: drop the file into a `data/` folder (or change `DATA_PATH`).

In [ ]:
# B.2 — Loading

raw = pd.read_csv(DATA_PATH)
print(f"Dimensions: {raw.shape[0]:,} rows × {raw.shape[1]} columns")
display(raw.head(5))

### B.3 Description of the variables
Each row = **one parcel to collect**. Here is what each variable is used for in our problem:

| Variable | Description | Role in the project |
|---|---|---|
| `order_id` | parcel identifier | key |
| `courier_id` | courier identifier | groups parcels into routes |
| `ds` | date (MMDD format) | groups into days; time-based split |
| `lng`, `lat` | coordinates of the pickup point | distance computation |
| `region_id`, `aoi_id`, `aoi_type` | zone, "area of interest" and its type (office, residential…) | ML explanatory variables |
| `time_window_start`, `time_window_end` | time window promised to the customer | deadline constraint of the optimisation |
| `accept_time` | moment the task becomes known to the courier | dynamics: a task cannot be planned before it exists |
| `pickup_time` | actual pickup moment | real sequence; ML target |
| `*_gps_*` | courier's GPS position at acceptance / at pickup | quality control only (often missing) |

In [ ]:
# B.3 — Types, descriptive statistics, missing values
print("Column types:")
print(raw.dtypes.to_string())

print("\nDescriptive statistics of the numeric variables:")
display(raw[["lng", "lat", "region_id", "aoi_id", "aoi_type"]].describe().round(4))

manq = raw.isna().sum()
manq = pd.DataFrame({"missing": manq, "percent": (100 * manq / len(raw)).round(1)})
print("\nMissing values per column:")
display(manq[manq.missing > 0])

print("Number of couriers:", raw.courier_id.nunique())
print("Number of regions :", raw.region_id.nunique(), "| number of AOIs:", raw.aoi_id.nunique())
print("Number of days    :", raw.ds.nunique(), f"(from {raw.ds.min()} to {raw.ds.max()}, MMDD format)")

### B.4 Data quality issues identified
* **No year in the dates** (`05-18 08:16:00`): we add a dummy year (2022) only to be able to manipulate dates. **Consequence:** the *day of the week* is unknown, so we do not use it as a variable.
* **Missing GPS columns** (≈ 21 % and 36 %): we do not use them in the model; the declared coordinates of the pickup point (`lng`, `lat`) are complete.
* **Time windows**: to be checked (length, actual compliance).
* **Progressive arrival of tasks** (`accept_time`): to be measured, since it determines how we optimise.

## C. Data Preprocessing

### C.1 Date conversion and consistency checks
*What we do:* convert text to dates, then check duplicates and time inconsistencies.
*Why:* a pickup before the task was accepted, or a window ending before it starts, would be a data error.

In [ ]:
# C.1 — Dates, duplicates, consistency

df = raw.copy()
TCOLS = ["accept_time", "time_window_start", "time_window_end", "pickup_time"]
for c in TCOLS:
    df[c] = pd.to_datetime("2022-" + df[c].astype(str), format="%Y-%m-%d %H:%M:%S")

journal = []   # preprocessing log, reused by the report and the website
def noter(etape, lignes, explication):
    journal.append({"etape": etape, "lignes": int(lignes), "explication": explication})
    print(f"[{etape}] {lignes:,} — {explication}")

noter("Raw data", len(df), "parcels loaded")
noter("Exact duplicates", df.duplicated().sum(), "identical rows")
noter("Duplicate identifiers", df.order_id.duplicated().sum(), "repeated order_id")
noter("Pickup before acceptance", (df.pickup_time < df.accept_time).sum(), "time inconsistency")
noter("Reversed window", (df.time_window_end <= df.time_window_start).sum(), "end before start")

duree_fen = (df.time_window_end - df.time_window_start).dt.total_seconds() / 60
print(f"\nWindow length: median {duree_fen.median():.0f} min, "
      f"{(duree_fen == 120).mean():.1%} of the windows are exactly 2 h")
print(f"Pickups BEFORE the window start: {(df.pickup_time < df.time_window_start).mean():.1%}")
print(f"Pickups AFTER the window end   : {(df.pickup_time > df.time_window_end).mean():.2%}")

**Reading:** no duplicates and no inconsistencies. Two important findings:
* Windows are almost always **2 hours** long.
* Couriers often collect **before** the start of the window, and very rarely **after** its end. The window **start** is therefore not a real constraint; the window **end** is a **deadline**.

**Modelling choice:** in the optimisation, the window end will be a penalised deadline; we will not force waiting for the window start (this reproduces the observed practice).

### C.2 Detecting geographic outliers
*Why:* a pickup point 500 km away from the city would distort the distances.

In [ ]:
# C.2 — Geographic outliers

q = df[["lng", "lat"]].quantile([0.001, 0.999])
print("Range of declared coordinates (0.1 % – 99.9 %):")
display(q.round(4))
hors_zone = ~df.lng.between(119.5, 122.0) | ~df.lat.between(36.5, 38.0)
noter("Points outside the Yantai area", hors_zone.sum(), "outlying coordinates (no removal needed if 0)")
df = df[~hors_zone].copy()

### C.3 Feature engineering: from parcels to *visits*
*What we do:* we sort each courier's parcels by pickup time. Several consecutive parcels at the **same place** are merged into a single **visit** (the courier does not move between them).

For each visit we compute:
* `t_debut` / `t_fin`: first and last pickup of the visit;
* `connue_a` (*release time*): moment when **all** its parcels are known (max of `accept_time`) — a cautious choice;
* `echeance`: the most binding window end among its parcels;
* `n_colis`: number of parcels collected.

In [ ]:
# C.3 — Building the visits

df = df.sort_values(["courier_id", "ds", "pickup_time", "order_id"]).reset_index(drop=True)
df["cle_lieu"] = df.lng.round(4).astype(str) + "_" + df.lat.round(4).astype(str)
nouvelle = df.cle_lieu != df.groupby(["courier_id", "ds"]).cle_lieu.shift()
df["visite"] = nouvelle.astype(int).groupby([df.courier_id, df.ds]).cumsum()

visites = (df.groupby(["courier_id", "ds", "visite"])
             .agg(lng=("lng", "first"), lat=("lat", "first"), region_id=("region_id", "first"),
                  aoi_id=("aoi_id", "first"), aoi_type=("aoi_type", "first"),
                  t_debut=("pickup_time", "min"), t_fin=("pickup_time", "max"),
                  connue_a=("accept_time", "max"), echeance=("time_window_end", "min"),
                  n_colis=("order_id", "size"))
             .reset_index())
visites["mois"] = visites.t_debut.dt.month
noter("Visits built", len(visites), "consecutive parcels at the same place merged")
print(f"Parcels per visit: mean {visites.n_colis.mean():.2f}")
display(visites.head())

### C.4 Building the "visit → next visit" pairs (training data)
The **ML target** is the **time between two consecutive visits**: from the end of visit *i* to the start of visit *j*, in minutes. It contains the travel **and** the service time at point *j*.

**Explanatory variables** (all known *before* leaving, hence no data leakage):
* `dist_km`: straight-line distance (Haversine formula);
* `meme_aoi`: are both points in the same area of interest?
* `aoi_type_dep`, `aoi_type_arr`: type of place at departure and arrival;
* `heure`: departure hour (traffic, breaks);
* `n_colis_arr`: number of parcels to collect on arrival (service time);
* `region_id`, `lat_arr`, `lng_arr`: location (neighbourhood).

**Causality filter:** a pair is kept only if the next visit was **already known** when the courier left the previous one. Otherwise the gap measures the wait for a new order, not a trip.

**Variables excluded to avoid leakage:** the actual pickup time of the target visit, the speed achieved on the same day, the GPS pickup positions.

In [ ]:
# C.4 — Consecutive pairs and explanatory variables

def haversine_km(lng1, lat1, lng2, lat2):
    """Straight-line distance between two points (km)."""
    lng1, lat1, lng2, lat2 = map(np.radians, (lng1, lat1, lng2, lat2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lng2 - lng1) / 2) ** 2
    return 6371.0 * 2 * np.arcsin(np.sqrt(a))

g = visites.groupby(["courier_id", "ds"])
prec = g.shift(1)                                   # the previous visit
paires = visites.copy()
paires["dist_km"] = haversine_km(prec.lng, prec.lat, visites.lng, visites.lat)
paires["meme_aoi"] = (prec.aoi_id == visites.aoi_id).astype(float)
paires["aoi_type_dep"] = prec.aoi_type
paires["aoi_type_arr"] = visites.aoi_type
paires["heure"] = prec.t_fin.dt.hour + prec.t_fin.dt.minute / 60
paires["n_colis_arr"] = visites.n_colis
paires["lat_arr"], paires["lng_arr"] = visites.lat, visites.lng
paires["ecart_min"] = (visites.t_debut - prec.t_fin).dt.total_seconds() / 60
paires["connue_avant_depart"] = visites.connue_a <= prec.t_fin
paires = paires[prec.t_fin.notna()].copy()
paires["aoi_type_dep"] = paires.aoi_type_dep.astype(int)

noter("Consecutive pairs", len(paires), "trips visit → next visit")
paires = paires[paires.connue_avant_depart].copy()
noter("Causal pairs", len(paires), "the next visit was already known at departure")
print(paires.ecart_min.describe(percentiles=[.5, .75, .9, .95, .99]).round(1).to_string())

### C.5 Handling target outliers
The distribution of gaps has a **long tail**: beyond 2 hours, the gap no longer corresponds to a trip but to a **break** or another activity (deliveries, return to the depot).

*Justified decision:* we **exclude from training** gaps > 120 minutes (≈ 2 % of the pairs). They are **not removed from the dataset**: they remain in the real days and in the real route durations.

In [ ]:
# C.5 — Breaks (gap > 120 min) excluded from training

SEUIL_PAUSE = 120
pause = paires.ecart_min > SEUIL_PAUSE
noter("Breaks excluded from training", pause.sum(), f"gaps > {SEUIL_PAUSE} min ({pause.mean():.1%} of pairs)")
apprentissage = paires[~pause].copy()
noter("Training pairs", len(apprentissage), "final base for ML")

### C.6 Train / test split — without data leakage
Two precautions:
1. **Temporal split:** the model learns on **May → August** and is tested on **September → October**. We never predict the past with the future.
2. **Two disjoint groups of couriers (A and B), drawn at random:**
   * The **planner model** (the one that helps decide) learns on couriers **A**.
   * An independent **evaluator model**, trained on couriers **B**, plays the role of "simulated reality" to score the routes. This way, **the model that decides is not the model that scores**: we avoid grading ourselves.

The days simulated in part F are **test** days (Sept.–Oct.) of couriers **A**. Neither model has seen them.

**Encoding and scaling:** trees (Random Forest, Gradient Boosting) need no scaling. Linear regression receives a *one-hot encoding* of the categories and standardised numeric variables.

In [ ]:
# C.6 — Temporal split and courier groups

coursiers = np.sort(apprentissage.courier_id.unique()) 

groupe_A = set(rng.choice(coursiers, size=len(coursiers) // 2, replace=False))
apprentissage["groupe"] = np.where(apprentissage.courier_id.isin(groupe_A), "A", "B")

FEATURES = ["dist_km", "meme_aoi", "aoi_type_dep", "aoi_type_arr", "heure",
            "n_colis_arr", "region_id", "lat_arr", "lng_arr"]
CATEGORIELLES = ["aoi_type_dep", "aoi_type_arr", "region_id"]
CIBLE = "ecart_min"

train = apprentissage[(apprentissage.groupe == "A") & (apprentissage.mois <= 8)]
test = apprentissage[(apprentissage.groupe == "A") & (apprentissage.mois >= 9)]
evaluateur_data = apprentissage[apprentissage.groupe == "B"]
X_train, y_train = train[FEATURES], train[CIBLE]
X_test, y_test = test[FEATURES], test[CIBLE]

print(f"Planner training (A, May–Aug)         : {len(train):,} pairs")
print(f"Test (A, Sept.–Oct.)                  : {len(test):,} pairs")
print(f"Independent evaluator (B, all months) : {len(evaluateur_data):,} pairs")

donnees_preparees = pd.concat([train.assign(jeu="train"), test.assign(jeu="test")])
donnees_preparees[["courier_id", "ds"] + FEATURES + [CIBLE, "jeu"]].sample(50000, random_state=SEED) \
    .to_csv(BASE / "data" / "donnees_preparees_echantillon.csv", index=False)

## D. Exploratory Data Analysis (EDA)
Each chart answers a **logistics question** that is useful for what follows.

### Figure 1 — Is the activity volume stable?

In [ ]:
# D.1 — Daily volume

vol = df.groupby(df.pickup_time.dt.normalize()).size()
fig, ax = plt.subplots(figsize=(10, 3.6))
ax.plot(vol.index, vol.values, color=C_AVEC, lw=1.6)
ax.axvline(pd.Timestamp("2022-09-01"), color=C_GRIS, ls="--", lw=1)
ax.text(pd.Timestamp("2022-09-03"), vol.max() * 0.95, "start of the test period", color=C_GRIS, fontsize=9)
ax.set(title="Figure 1 — Parcels collected per day in Yantai", xlabel="Date (May → October)", ylabel="Parcels per day")
sauver(fig, "fig01_volume_quotidien"); plt.show()
print(f"Mean: {vol.mean():.0f} parcels/day; minimum {vol.min()}; maximum {vol.max()}")

**Interpretation:** activity rises in May, peaks in June (two peaks close to 9,500 parcels/day), then stabilises around 6,000 parcels/day in July–August. The test period (September–October) oscillates between 4,700 and 7,500 parcels/day: it stays comparable to the training period, which makes the model test representative.

### Figure 2 — How big is a route?

In [ ]:
# D.2 — Visits per courier-day

taille = visites.groupby(["courier_id", "ds"]).size()
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.hist(taille.clip(upper=60), bins=60, color=C_AVEC, edgecolor="white")
ax.axvspan(10, 30, color=C_SANS, alpha=0.12, label="days kept for optimisation (10–30 visits)")
ax.set(title="Figure 2 — Number of visits per courier day", xlabel="Visits in the day (capped at 60)",
       ylabel="Number of courier-days")
ax.legend(); sauver(fig, "fig02_taille_tournees"); plt.show()
print(f"{len(taille):,} courier-days; median {taille.median():.0f} visits; "
      f"{((taille >= 10) & (taille <= 30)).mean():.0%} have between 10 and 30 visits")

**Interpretation:** many days have only one or two visits (part-time couriers), while a large share of days contain **10 to 30 visits**. This is where the visiting order really matters: we will optimise these days.

### Figure 3 — How long between two visits?

In [ ]:
# D.3 — Target distribution

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.hist(paires.ecart_min.clip(upper=200), bins=100, color=C_AVEC, edgecolor="white")
ax.axvline(SEUIL_PAUSE, color=C_SANS, lw=2, label=f"'break' threshold = {SEUIL_PAUSE} min")
ax.set(title="Figure 3 — Time between two consecutive visits", xlabel="Gap (minutes, capped at 200)",
       ylabel="Number of trips"); ax.legend()
sauver(fig, "fig03_distribution_ecarts"); plt.show()
print(f"Median: {paires.ecart_min.median():.0f} min; 75 % of trips last less than {paires.ecart_min.quantile(.75):.0f} min")

**Interpretation:** most trips last about ten minutes; the tail beyond 2 h corresponds to breaks, excluded from training (C.5). The observed minimum is 5 minutes, which reflects a minimal service time per visit.

### Figure 4 — Does distance explain time?

In [ ]:
# D.4 — Time vs distance

bornes = [0, 0.05, 0.2, 0.5, 1, 2, 3, 5, 8, 15, 100]
noms_bandes = ["<50 m", "50–200 m", "0.2–0.5 km", "0.5–1 km", "1–2 km", "2–3 km", "3–5 km", "5–8 km", "8–15 km", ">15 km"]
apprentissage["bande_dist"] = pd.cut(apprentissage.dist_km, bornes, labels=noms_bandes, include_lowest=True)
stat_dist = apprentissage.groupby("bande_dist", observed=False).ecart_min.describe(percentiles=[.25, .5, .75])
fig, ax = plt.subplots(figsize=(9, 3.8))
x = np.arange(len(stat_dist))
ax.fill_between(x, stat_dist["25%"], stat_dist["75%"], color=C_AVEC, alpha=0.18, label="central 50 % of trips")
ax.plot(x, stat_dist["50%"], "o-", color=C_AVEC, lw=2, label="median")
ax.set_xticks(x, noms_bandes, rotation=30)
ax.set(title="Figure 4 — Time between visits by distance", xlabel="Distance between the two points",
       ylabel="Time (min)"); ax.legend()
sauver(fig, "fig04_temps_vs_distance"); plt.show()
display(stat_dist[["count", "50%"]].rename(columns={"count": "trips", "50%": "median (min)"}).round(1))

**Interpretation:** the median time goes from 8 min (under 50 m) to 13 min (1 to 5 km), then **decreases** for long trips, probably made by vehicle on fast roads. The relationship is therefore **neither proportional nor linear**: even under 50 m it takes several minutes (service, access, waiting for the customer), and for the same distance the time varies enormously (blue band). A "distance ÷ speed" formula does not capture this shape; this is what ML has to learn.

### Figure 5 — Does the time of day matter?

In [ ]:
# D.5 — Effect of the hour

par_heure = apprentissage.assign(h=apprentissage.heure.astype(int)).query("6 <= h <= 21").groupby("h").ecart_min.agg(["median", "mean", "size"])
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.bar(par_heure.index, par_heure["mean"], color=C_AVEC, width=0.7, label="mean")
ax.plot(par_heure.index, par_heure["median"], "o-", color=C_SANS, lw=2, label="median")
ax.set(title="Figure 5 — Time between visits by departure hour", xlabel="Hour of the day",
       ylabel="Time (min)"); ax.legend()
sauver(fig, "fig05_effet_heure"); plt.show()

**Interpretation:** in the morning (7 am – 10 am) trips are fast (about 11 to 13 min on average); between 11 am and 4 pm, the average time rises to about 20 min, with a peak at 4 pm. The median barely moves: it is mainly **long or interrupted trips** that become more frequent in the afternoon. **Consequence for the optimisation:** the time matrix must depend on the hour. ML will therefore provide a matrix **per hour**: the same route does not have the same cost at 9 am and at 4 pm.

### Figure 6 — Does the type of place influence the time?

In [ ]:
# D.6 — AOI type

par_type = apprentissage.groupby("aoi_type_arr").ecart_min.agg(["median", "mean", "size"]).sort_values("size", ascending=False).head(8)
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.bar([f"type {i}" for i in par_type.index], par_type["mean"], color=C_AVEC, width=0.65)
for i, (m, n) in enumerate(zip(par_type["mean"], par_type["size"])):
    ax.text(i, m + 0.3, f"{n/1000:.0f} k", ha="center", fontsize=8, color=C_GRIS)
ax.set(title="Figure 6 — Mean time by arrival place type (8 most frequent types)",
       xlabel="AOI type (area of interest) — label = number of trips", ylabel="Mean time (min)")
sauver(fig, "fig06_type_aoi"); plt.show()

**Interpretation:** the differences between place types are **modest** (14 to 16 min on average). The AOI type brings some information, but a secondary one: we keep it as an explanatory variable, and Figure 13 will confirm its low weight.

### Figure 7 — Are time windows respected?

In [ ]:
# D.7 — Position of the pickup within the window

marge = (df.time_window_end - df.pickup_time).dt.total_seconds() / 60
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.hist(marge.clip(-120, 400), bins=104, color=C_AVEC, edgecolor="white")
ax.axvline(0, color=C_SANS, lw=2, label="window end (deadline)")
ax.axvline(120, color=C_GRIS, lw=1.2, ls="--", label="window start (2 h window)")
ax.set(title="Figure 7 — Margin left before the deadline at pickup time",
       xlabel="Minutes before the window end (negative = late; values capped to [−120; 400])", ylabel="Parcels"); ax.legend()
sauver(fig, "fig07_fenetres_horaires"); plt.show()
print(f"Late: {(marge < 0).mean():.2%}; collected before the window start: {(marge > duree_fen).mean():.1%}")

**Interpretation:** real couriers **almost always meet the deadline** (less than 1 % late), often with a wide margin, and one third of the parcels are collected **before** the start of the window. The gain to expect from optimisation therefore concerns **time and distance**, without degrading window compliance (which will be monitored as a safeguard).

### Figure 8 — Are the tasks known from the morning?

In [ ]:
# D.8 — Progressive arrival of tasks

premier = visites.groupby(["courier_id", "ds"]).t_debut.transform("min")
avance = (visites.connue_a - premier).dt.total_seconds() / 3600
fig, ax = plt.subplots(figsize=(8, 3.6))
vals = np.sort(avance.clip(-3, 12)); ax.plot(vals, np.linspace(0, 1, len(vals)), color=C_AVEC, lw=2)
ax.axvline(0, color=C_SANS, lw=1.5, label="first pickup of the day")
ax.set(title="Figure 8 — Moment when visits become known to the courier",
       xlabel="Hours after the first pickup of the day", ylabel="Cumulative share of visits")
ax.legend(); sauver(fig, "fig08_arrivee_taches"); plt.show()
print(f"Visits already known before the first pickup: {(avance <= 0).mean():.0%}")

**Interpretation:** fewer than half of the visits are known before the first pickup; the others arrive during the day. **This is a major methodological finding:** planning the whole day from the morning would be unrealistic (it would be "cheating" by using future information). We will therefore optimise **dynamically**: at each stop, the courier re-plans among the tasks already known only.

### Figure 9 — Where are the pickup points?

In [ ]:
# D.9 — Map of pickup points

ech = visites.sample(60000, random_state=SEED)
fig, ax = plt.subplots(figsize=(7, 5.2))
ax.scatter(ech.lng, ech.lat, s=0.6, c=ech.region_id, cmap="viridis", alpha=0.5)
ax.set(title="Figure 9 — Pickup points in Yantai (colour = region)", xlabel="Longitude", ylabel="Latitude")
ax.set_aspect(1 / np.cos(np.radians(37.4)))
sauver(fig, "fig09_carte_points"); plt.show()

**Interpretation:** pickups are concentrated in several separate urban hubs, each served by different regions. Distances between two points of the same route are therefore mostly **local**; the geography of the area (`region_id`, coordinates) is useful information for the model.

### Figure 10 — Correlations between variables

In [ ]:
# D.10 — Correlations
cols_corr = ["dist_km", "meme_aoi", "heure", "n_colis_arr", "lat_arr", "lng_arr", "ecart_min"]
corr = apprentissage[cols_corr].sample(200000, random_state=SEED).corr(method="spearman")
fig, ax = plt.subplots(figsize=(6.4, 5.2))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(cols_corr)), cols_corr, rotation=45, ha="right"); ax.set_yticks(range(len(cols_corr)), cols_corr)
for i in range(len(cols_corr)):
    for j in range(len(cols_corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if abs(corr.iloc[i, j]) > 0.5 else "black")
ax.grid(False); fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Figure 10 — Spearman correlations")
sauver(fig, "fig10_correlations"); plt.show()

**Interpretation:** distance (0.27) and hour (0.17) are the variables most related to time; being in the same AOI goes the other way (−0.18: short trips). All these correlations remain **weak**: no single variable explains the time. This is an argument for a **non-linear** model that combines several variables, and a sign that prediction will remain difficult.

In [ ]:
# D.11 — Export of the EDA data for the web platform
hist_ecarts, bords = np.histogram(paires.ecart_min.clip(upper=200), bins=40, range=(0, 200))
hist_marge, bords_m = np.histogram(marge.clip(-120, 400), bins=52, range=(-120, 400))
hist_taille, bords_t = np.histogram(taille.clip(upper=60), bins=60, range=(1, 61))
eda = {
    "volume_quotidien": {"dates": [d.strftime("%m-%d") for d in vol.index], "colis": vol.astype(int).tolist()},
    "hist_ecarts": {"bords": bords.tolist(), "valeurs": hist_ecarts.tolist()},
    "temps_vs_distance": {"bandes": noms_bandes, "q25": stat_dist["25%"].round(2).tolist(),
                          "mediane": stat_dist["50%"].round(2).tolist(), "q75": stat_dist["75%"].round(2).tolist(),
                          "n": stat_dist["count"].astype(int).tolist()},
    "par_heure": {"heures": par_heure.index.tolist(), "moyenne": par_heure["mean"].round(2).tolist(),
                  "mediane": par_heure["median"].round(2).tolist(), "n": par_heure["size"].astype(int).tolist()},
    "par_type_aoi": {"types": [int(i) for i in par_type.index], "moyenne": par_type["mean"].round(2).tolist(),
                     "n": par_type["size"].astype(int).tolist()},
    "hist_marge": {"bords": bords_m.tolist(), "valeurs": hist_marge.tolist()},
    "hist_taille": {"bords": bords_t.tolist(), "valeurs": hist_taille.tolist()},
    "correlations": {"variables": cols_corr, "matrice": corr.round(3).values.tolist()},
    "points_carte": ech.sample(4000, random_state=SEED)[["lng", "lat", "region_id"]].round(5).values.tolist(),
    "indicateurs": {
        "colis": int(len(raw)), "colonnes": int(raw.shape[1]), "coursiers": int(raw.courier_id.nunique()),
        "regions": int(raw.region_id.nunique()), "aoi": int(raw.aoi_id.nunique()), "jours": int(raw.ds.nunique()),
        "journees_coursier": int(len(taille)), "visites": int(len(visites)),
        "paires_causales": int(len(paires)), "paires_apprentissage": int(len(apprentissage)),
        "taux_retard_reel": float((marge < 0).mean()), "taux_avant_fenetre": float((marge > duree_fen).mean()),
        "part_connue_avant_debut": float((avance <= 0).mean()), "ecart_median": float(paires.ecart_min.median()),
        "fenetre_mediane_min": float(duree_fen.median()),
    },
    "valeurs_manquantes": {c: int(v) for c, v in raw.isna().sum().items()},
    "journal_preprocessing": journal,
}
print("Key indicators exported:", json.dumps(eda["indicateurs"], indent=1, ensure_ascii=False))

## E. Machine Learning

### E.1 Problem definition
* **y (target)**: `ecart_min`, the time in minutes between the end of one visit and the start of the next.
* **X**: the 9 variables presented in C.4.
* **Problem type**: regression; metrics **MAE** (average error in minutes, the most telling), **RMSE** (penalises large errors) and **R²** (share of variability explained).

### E.2 Baseline: the "traditional" formula
This is what a planner without ML would use, based on **historical averages**:

**time = median service time + distance ÷ median speed**

Both parameters are estimated on the training data only.

In [ ]:
# E.2 — "Historical" baseline (formula without ML)

SERVICE_MED = float(train.loc[train.dist_km < 0.05, CIBLE].median())          # minutes
loin = train[train.dist_km > 0.3]
VITESSE_MED = float(np.median(loin.dist_km / np.maximum(loin[CIBLE] - SERVICE_MED, 1.0)))   # km/min

def prediction_baseline(dist_km):
    return SERVICE_MED + np.asarray(dist_km) / VITESSE_MED

print(f"Median service time: {SERVICE_MED:.1f} min")
print(f"Median speed       : {VITESSE_MED * 60:.1f} km/h")

### E.3 Candidate models
We compare, from the simplest to the most flexible:
1. **Linear regression** — simple and explainable: one coefficient per variable.
2. **Decision tree** (depth 10) — readable "if… then…" rules.
3. **Random Forest** — average of many trees: robust.
4. **Gradient Boosting** (`HistGradientBoostingRegressor`) — trees built successively, each correcting the errors of the previous one; fast on large volumes.

The Random Forest is trained on a sample of 150,000 pairs to stay fast in Colab; the other models use all training pairs.

In [ ]:
# E.3 — Training and comparison

num = [f for f in FEATURES if f not in CATEGORIELLES]
lineaire = make_pipeline(
    ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORIELLES),
                       ("num", StandardScaler(), num)]),
    LinearRegression())
modeles = {
    "Linear regression": lineaire,
    "Decision tree": DecisionTreeRegressor(max_depth=10, min_samples_leaf=50, random_state=SEED),
    "Random Forest": RandomForestRegressor(n_estimators=60, max_depth=16, min_samples_leaf=20,
                                           n_jobs=-1, random_state=SEED),
    "Gradient Boosting": HistGradientBoostingRegressor(max_iter=300, learning_rate=0.08, max_leaf_nodes=63,
                                                       categorical_features=[FEATURES.index(c) for c in CATEGORIELLES],
                                                       random_state=SEED),
}

def evaluer(nom, y_vrai, y_pred, duree=0.0):
    return {"modele": nom, "MAE": mean_absolute_error(y_vrai, y_pred),
            "Biais": float(np.mean(np.asarray(y_pred) - np.asarray(y_vrai))),
            "RMSE": math.sqrt(mean_squared_error(y_vrai, y_pred)), "R2": r2_score(y_vrai, y_pred),
            "temps_entrainement_s": duree}

resultats, predictions = [], {}
predictions["Baseline (historical averages)"] = prediction_baseline(X_test.dist_km)
resultats.append(evaluer("Baseline (historical averages)", y_test, predictions["Baseline (historical averages)"]))
for nom, modele in modeles.items():
    t0 = time.time()
    if nom == "Random Forest":
        idx = train.sample(150000, random_state=SEED).index
        modele.fit(X_train.loc[idx], y_train.loc[idx])
    else:
        modele.fit(X_train, y_train)
    duree = time.time() - t0
    predictions[nom] = modele.predict(X_test)
    resultats.append(evaluer(nom, y_test, predictions[nom], duree))
    print(f"{nom:<22} trained in {duree:5.1f} s")

tableau_ml = pd.DataFrame(resultats).set_index("modele")
display(tableau_ml.round(3))
print(f"Mean actual time between two visits (test): {y_test.mean():.2f} min; median: {y_test.median():.0f} min")

**Reading the table — a counter-intuitive but instructive result:**
* The **baseline has the smallest MAE**: it predicts a "typical" time (close to the median, 9 min). Since most trips are short, it is rarely far off *in general*.
* But it has a **strongly negative bias**: on average, it **underestimates** each trip by several minutes, because it ignores the long tail of slow trips (Figure 3).
* The ML models have a slightly higher MAE, but a **lower RMSE**, a **positive R²** and an **almost zero bias**: on average, they predict correctly.

**Why bias matters more than MAE for our decision:** a route adds up about twenty trips. An underestimation of a few minutes per trip becomes **more than an hour of error** at the end of the day: an "over-optimistic" plan misses deadlines. For optimisation, we therefore need **unbiased** times.

**Selection criterion:** among the ML models, we keep the one with the **lowest RMSE** (and therefore the best R²), provided its bias is close to zero.

> R² remains modest (≈ 0.1): a large part of the time between two visits depends on elements absent from the data (customer waiting, phone call, deliveries made between two pickups). This is an honest limit of the dataset, not a methodological error.

### E.4 Visual comparison of performance

In [ ]:
# E.4 — Model comparison chart

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True)
couleurs = [C_SANS] + [C_REEL] * 3 + [C_AVEC]
for ax, col, titre in zip(axes, ["MAE", "RMSE", "Biais"],
                          ["MAE (min) — smaller = better", "RMSE (min) — smaller = better", "Mean bias (min) — closer to 0 = better"]):
    vals = tableau_ml[col]
    ax.barh(tableau_ml.index, vals, color=couleurs, height=0.6)
    for i, v in enumerate(vals):
        ax.text(v, i, f" {v:.2f}", va="center", ha="left" if v >= 0 else "right", fontsize=9)
    ax.set_title(titre, fontsize=10); ax.axvline(0, color=C_GRIS, lw=0.8)
    lo, hi = min(0, vals.min()), vals.max()
    ax.set_xlim(lo * 1.45 if lo < 0 else 0, hi * 1.3 if hi > 0 else 1)
axes[0].invert_yaxis()
fig.suptitle("Figure 11 — Model comparison on the test set (Sept.–Oct.)", fontweight="bold")
fig.tight_layout(); sauver(fig, "fig11_comparaison_modeles"); plt.show()

### E.5 Error analysis
Where does the model go wrong most? We compare the error of the best model and of the baseline **by distance range**.

In [ ]:
# E.5 — Errors by distance range

NOM_FINAL = tableau_ml.drop(index="Baseline (historical averages)").RMSE.idxmin()
modele_final = modeles[NOM_FINAL]
err = pd.DataFrame({"bande": test.bande_dist if "bande_dist" in test else pd.cut(test.dist_km, bornes, labels=noms_bandes, include_lowest=True),
                    "ml": np.abs(predictions[NOM_FINAL] - y_test),
                    "base": np.abs(predictions["Baseline (historical averages)"] - y_test)})
err_bande = err.groupby("bande", observed=False)[["base", "ml"]].mean()
fig, ax = plt.subplots(figsize=(9, 3.6))
x = np.arange(len(err_bande)); w = 0.38
ax.bar(x - w / 2, err_bande.base, w, color=C_SANS, label="Baseline (formula)")
ax.bar(x + w / 2, err_bande.ml, w, color=C_AVEC, label=NOM_FINAL)
ax.set_xticks(x, err_bande.index, rotation=30)
ax.set(title="Figure 12 — Mean error by trip distance", xlabel="Distance", ylabel="MAE (min)"); ax.legend()
sauver(fig, "fig12_erreurs_par_distance"); plt.show()
print("Selected model:", NOM_FINAL)
biais_route = 20 * tableau_ml.loc["Baseline (historical averages)", "Biais"]
print(f"Expected cumulated baseline error on a 20-trip route : {biais_route:.0f} min")
print(f"Expected cumulated error of the selected model over 20 trips : {20 * tableau_ml.loc[NOM_FINAL, 'Biais']:+.0f} min")

**Reading:** on short trips (under 2 km, the vast majority), the formula does as well as or slightly better than ML, because it predicts the "typical" trip. Beyond 5 km, the formula is very wrong: it applies a bicycle speed to trips made by vehicle, and its error exceeds 100 min beyond 15 km. ML stays stable, around 15 to 20 min. **ML is mainly useful to avoid large errors**, and these are precisely the ones that distort a route.

### E.6 Interpretation: which variables matter?
We use **permutation importance**: we shuffle a variable at random and measure how much the error increases. The more the error increases, the more important the variable.

In [ ]:
# E.6 — Variable importance (permutation)

ech_test = test.sample(20000, random_state=SEED)
imp = permutation_importance(modele_final, ech_test[FEATURES], ech_test[CIBLE], n_repeats=5,
                             random_state=SEED, scoring="neg_mean_absolute_error", n_jobs=1)
importances = pd.Series(imp.importances_mean, index=FEATURES).sort_values()
noms_lisibles = {"dist_km": "distance (km)", "meme_aoi": "same AOI", "aoi_type_dep": "AOI type (departure)",
                 "aoi_type_arr": "AOI type (arrival)", "heure": "departure hour", "n_colis_arr": "parcels on arrival",
                 "region_id": "region", "lat_arr": "arrival latitude", "lng_arr": "arrival longitude"}
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.barh([noms_lisibles[f] for f in importances.index], importances.values, color=C_AVEC, height=0.6)
ax.set(title=f"Figure 13 — Variable importance ({NOM_FINAL})",
       xlabel="Increase in MAE when the variable is shuffled (min)")
sauver(fig, "fig13_importance_variables"); plt.show()

**Reading:** **location** (region, latitude, longitude) and **hour** weigh the most, ahead of distance itself. The model has learned that some neighbourhoods are slower than others (density, access, traffic), and that the afternoon is slower. The AOI type matters little, and the number of parcels on arrival does not matter: almost every visit contains a single parcel (1.01 on average).

### E.7 Final model selection and evaluator model
The chosen model is the one with the **lowest MAE on the test set**. It will serve as the **planner** in the optimisation.

We then train, with the same algorithm, the **evaluator model** on couriers B. It is only used to **score** routes ("simulated reality"). We also check its accuracy on the test set.

In [ ]:
# E.7 — Independent evaluator model

evaluateur = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.08, max_leaf_nodes=63,
                                           categorical_features=[FEATURES.index(c) for c in CATEGORIELLES],
                                           random_state=SEED + 1)
evaluateur.fit(evaluateur_data[FEATURES], evaluateur_data[CIBLE])
mae_eval = mean_absolute_error(y_test, evaluateur.predict(X_test))
print(f"MAE of the evaluator (couriers B) on the test set of couriers A: {mae_eval:.2f} min")

resultats_ml = {
    "modeles": tableau_ml.reset_index().round(4).to_dict(orient="records"),
    "modele_final": NOM_FINAL,
    "baseline": {"service_median_min": SERVICE_MED, "vitesse_mediane_kmh": VITESSE_MED * 60},
    "importances": {noms_lisibles[k]: float(v) for k, v in importances.sort_values(ascending=False).items()},
    "erreurs_par_distance": {"bandes": [str(b) for b in err_bande.index], "baseline": err_bande.base.round(3).tolist(),
                             "ml": err_bande.ml.round(3).tolist()},
    "evaluateur_mae": mae_eval,
    "tailles": {"train": int(len(train)), "test": int(len(test)), "evaluateur": int(len(evaluateur_data))},
    "features": FEATURES,
}

## F. Logistics optimisation — the core part

**Logic:** Data → ML model → **predicted time matrix** → **visiting-order optimisation** → gain evaluation.

### F.1 The mathematical model (static problem at a given instant)
At an instant τ, the courier is at point 0 and knows a set *N* of pending visits. We look for the visiting order that minimises:

$$\min \; \sum_{i}\sum_{j} \hat t_{ij}\,x_{ij} \;+\; \lambda \sum_{j \in N} L_j$$

subject to the constraints:

$$\sum_{i} x_{ij} = 1 \;\; \forall j \in N \qquad \text{(each visit is entered once)}$$
$$\sum_{j} x_{ij} = 1 \;\; \forall i \in N \cup \{0\} \qquad \text{(each visit is left once — the last arc goes to a dummy end node)}$$
$$a_j \;\ge\; a_i + \hat t_{ij} - M\,(1 - x_{ij}) \qquad \text{(time propagation; also eliminates subtours)}$$
$$L_j \;\ge\; a_j - l_j,\quad L_j \ge 0,\quad a_0 = \tau,\quad x_{ij} \in \{0,1\}$$

* **Decision variables:** $x_{ij}$ (go from *i* to *j*: yes/no), $a_j$ (arrival time), $L_j$ (lateness).
* **Parameters:** $\hat t_{ij}$ = time **predicted by ML**; $l_j$ = deadline (window end); λ = cost of one minute of lateness, expressed in minutes of travel.
* **Type:** **mixed-integer linear program (MILP)**, a variant of the **TSP with time windows (TSPTW)** — NP-hard.

### F.2 The dynamic version (realistic)
Since tasks arrive during the day (Figure 8), we solve this problem **at every stop** (*rolling horizon*):
1. the courier finishes a visit at instant τ;
2. we build the list of visits **already known** and not yet done;
3. if it is empty, the courier waits for the next task;
4. otherwise, we optimise the order of these visits; the courier heads to the **first** one in the list;
5. we start over at the next stop, with the new tasks that may have arrived.

### F.3 Solution methods
* **Heuristic** (used at each stop, very fast): **greedy** construction (go to the visit that costs the least right now), then **2-opt** local improvement (reverse a segment of the route) and **or-opt** (move a visit elsewhere), repeated until no improvement is possible.
* **Exact MILP** with the **HiGHS** solver (`scipy.optimize.milp`) on instances of 8 visits, to **measure the gap** between the heuristic and the proven optimum.

The time matrix depends on the hour (Figure 5): $\hat t_{ij}$ is read from the matrix of the departure hour.

In [ ]:
# F.3 — Optimisation heuristic

H0, H1 = 6, 22                     # hourly matrices from 6 am to 10 pm

def idx_heure(tau):
    """tau in minutes since midnight -> index of the hourly matrix."""
    return min(max(int(tau // 60), H0), H1) - H0

def cout_route(route, depart, tau, M, ech, lam):
    """Cost = sum of predicted times + lam × lateness, following the clock."""
    t, cout, prec = tau, 0.0, depart
    for j in route:
        d = M[idx_heure(t)][prec][j]
        t += d
        cout += d + lam * max(0.0, t - ech[j])
        prec = j
    return cout

def planifier(depart, tau, attente, M, ech, lam, max_passes=50):
    """Greedy + 2-opt + or-opt (first-improvement local search)."""
    route, restant, t, prec = [], list(attente), tau, depart
    while restant:                                      # 1) greedy construction
        meilleur, mcout = None, float("inf")
        for j in restant:
            d = M[idx_heure(t)][prec][j]
            c = d + lam * max(0.0, t + d - ech[j])
            if c < mcout - 1e-12:
                meilleur, mcout = j, c
        route.append(meilleur); restant.remove(meilleur)
        t += M[idx_heure(t)][prec][meilleur]; prec = meilleur
    best = cout_route(route, depart, tau, M, ech, lam)
    n = len(route)
    for _ in range(max_passes):                         # 2) local improvement
        ameliore = False
        for i in range(n - 1):                          # 2-opt: reverse route[i..k]
            for k in range(i + 1, n):
                cand = route[:i] + route[i:k + 1][::-1] + route[k + 1:]
                c = cout_route(cand, depart, tau, M, ech, lam)
                if c < best - 1e-9:
                    route, best, ameliore = cand, c, True
        for i in range(n):                              # or-opt: move one visit
            for k in range(n):
                if k == i:
                    continue
                cand = route[:i] + route[i + 1:]
                cand.insert(k, route[i])
                c = cout_route(cand, depart, tau, M, ech, lam)
                if c < best - 1e-9:
                    route, best, ameliore = cand, c, True
        if not ameliore:
            break
    return route, best

print("Heuristic ready.")

### F.4 Selecting test days and building the matrices
We keep **test** days (Sept.–Oct.) of couriers **A**, with 10 to 30 visits. For each one, we compute three hourly matrices (6 am → 10 pm):
* **without ML**: the baseline formula (median service + distance ÷ median speed);
* **with ML**: the predictions of the planner model;
* **evaluator**: the predictions of the independent model (couriers B), used **only to score** the routes.

In [ ]:
# F.4 — Test days and time matrices
N_JOURNEES = 150
taille_v = visites.groupby(["courier_id", "ds"]).size().rename("n").reset_index()
cand = taille_v[(taille_v.n >= 10) & (taille_v.n <= 30) & taille_v.courier_id.isin(groupe_A)
                & (taille_v.ds >= 901)]
journees = cand.sample(N_JOURNEES, random_state=SEED).sort_values(["ds", "courier_id"])[["courier_id", "ds"]]
print(f"Candidate days: {len(cand)}; simulated days: {len(journees)}")

HEURES = np.arange(H0, H1 + 1)

def matrices_journee(v):
    """Returns (without_ml, with_ml, evaluator) of shape (hours, n, n), in minutes."""
    n = len(v)
    I, J = np.meshgrid(np.arange(n), np.arange(n), indexing="ij")
    I, J = I.ravel(), J.ravel()
    dist = haversine_km(v.lng.values[I], v.lat.values[I], v.lng.values[J], v.lat.values[J])
    base = prediction_baseline(dist).reshape(n, n)
    lignes = []
    for h in HEURES:
        lignes.append(pd.DataFrame({
            "dist_km": dist, "meme_aoi": (v.aoi_id.values[I] == v.aoi_id.values[J]).astype(float),
            "aoi_type_dep": v.aoi_type.values[I], "aoi_type_arr": v.aoi_type.values[J], "heure": h + 0.5,
            "n_colis_arr": v.n_colis.values[J], "region_id": v.region_id.values[J],
            "lat_arr": v.lat.values[J], "lng_arr": v.lng.values[J]}))
    Xm = pd.concat(lignes, ignore_index=True)[FEATURES]
    ml = modele_final.predict(Xm).reshape(len(HEURES), n, n)
    ev = evaluateur.predict(Xm).reshape(len(HEURES), n, n)
    sans = np.repeat(base[None], len(HEURES), axis=0)
    for M in (ml, ev, sans):
        M[:, np.arange(n), np.arange(n)] = 0.0
        np.maximum(M, 1.0, out=M, where=~np.eye(n, dtype=bool)[None].repeat(len(HEURES), 0))
    dist_mat = np.round(dist.reshape(n, n), 4)
    # Rounded to 0.01 min: these are EXACTLY the values the web simulator will use
    return np.round(sans, 2), np.round(ml, 2), np.round(ev, 2), dist_mat

instances = []
t0 = time.time()
for cid, ds in journees.itertuples(index=False):
    v = visites[(visites.courier_id == cid) & (visites.ds == ds)].sort_values("visite").reset_index(drop=True)
    minuit = v.t_debut.iloc[0].normalize()
    m = lambda s: ((s - minuit).dt.total_seconds() / 60).values
    sans, ml, ev, dist_mat = matrices_journee(v)
    instances.append({"courier_id": int(cid), "ds": int(ds), "n": len(v), "v": v,
                      "connue": m(v.connue_a), "echeance": m(v.echeance), "debut": m(v.t_debut), "fin": m(v.t_fin),
                      "M_sans": sans.tolist(), "M_ml": ml.tolist(), "M_eval": ev.tolist(), "dist": dist_mat.tolist()})
print(f"Matrices computed for {len(instances)} days in {time.time() - t0:.0f} s")

### F.5 Simulation: real vs without ML vs with ML
The three strategies start from the **same first visit, at the same time** (the one from reality).
* **Real**: we replay the courier's actual order.
* **Without ML / With ML**: at each stop, the heuristic re-plans with its own time matrix.

In all cases, **the time that actually elapses is given by the independent evaluator**, and no visit can be made before it is known. The **distance** is computed exactly (Haversine): it depends on no model.

**Penalty assumption:** λ = 5 (one minute of lateness "costs" as much as 5 minutes of travel). A sensitivity analysis follows in F.8.

In [ ]:
# F.5 — Dynamic simulator
LAMBDA = 5.0

def simuler(inst, strategie, lam=LAMBDA, toutes_connues=False):
    """strategie: 'reel', 'sans_ml' or 'avec_ml'. Returns the KPIs and the order followed."""
    E, D, ech = inst["M_eval"], inst["dist"], inst["echeance"]
    connue = np.full(inst["n"], -1e9) if toutes_connues else inst["connue"]
    Mplan = inst["M_sans"] if strategie == "sans_ml" else inst["M_ml"]
    n = inst["n"]
    cur, tau = 0, float(inst["fin"][0])          # common start: real end of the 1st visit
    t_start = tau
    faites, ordre = {0}, [0]
    deplacement = attente = distance = retard_min = 0.0
    n_retard = 1 if inst["fin"][0] > ech[0] else 0
    reel_suivant = iter(range(1, n))
    while len(faites) < n:
        if strategie == "reel":
            nxt = next(reel_suivant)
            if connue[nxt] > tau:
                attente += connue[nxt] - tau; tau = float(connue[nxt])
        else:
            dispo = [j for j in range(n) if j not in faites and connue[j] <= tau]
            if not dispo:
                prochaine = min(connue[j] for j in range(n) if j not in faites)
                attente += prochaine - tau; tau = float(prochaine)
                continue
            route, _ = planifier(cur, tau, dispo, Mplan, ech, lam)
            nxt = route[0]
        d = E[idx_heure(tau)][cur][nxt]
        tau += d; deplacement += d; distance += D[cur][nxt]
        if tau > ech[nxt]:
            n_retard += 1; retard_min += tau - ech[nxt]
        faites.add(nxt); ordre.append(nxt); cur = nxt
    return {"deplacement_min": deplacement, "attente_min": attente, "duree_min": tau - t_start,
            "distance_km": distance, "n_retard": n_retard, "retard_min": retard_min, "ordre": ordre}

t0 = time.time()
lignes = []
for k, inst in enumerate(instances):
    res = {s: simuler(inst, s) for s in ("reel", "sans_ml", "avec_ml")}
    inst["resultats"] = res
    vrai = inst["fin"][-1] - inst["fin"][0]
    for s, r in res.items():
        lignes.append({"journee": k, "courier_id": inst["courier_id"], "ds": inst["ds"], "n": inst["n"],
                       "strategie": s, "duree_reelle_observee": vrai,
                       **{c: r[c] for c in ("deplacement_min", "attente_min", "duree_min", "distance_km", "n_retard", "retard_min")}})
sim = pd.DataFrame(lignes)
print(f"Simulation of {len(instances)} days × 3 strategies in {time.time() - t0:.0f} s")

### F.6 Evaluator validation: is the "simulated reality" credible?
For the **real** order, we compare the duration **simulated** by the evaluator with the duration **actually observed** in the data.

In [ ]:
# F.6 — Anchoring in reality
reel = sim[sim.strategie == "reel"]
ratio = reel.duree_min.sum() / reel.duree_reelle_observee.sum()
corr_duree = np.corrcoef(reel.duree_min, reel.duree_reelle_observee)[0, 1]
err_duree = np.median(np.abs(reel.duree_min - reel.duree_reelle_observee) / reel.duree_reelle_observee)
print(f"Simulated duration / observed duration (total): {ratio:.2f}")
print(f"Correlation per day                     : {corr_duree:.2f}")
print(f"Median relative error per day           : {err_duree:.1%}")
retards_obs = [int((inst["fin"] > inst["echeance"]).sum()) for inst in instances]
print(f"Late pickups per day — observed in the data: {np.mean(retards_obs):.2f}; simulated for the real order: {reel.n_retard.mean():.2f}")
fig, ax = plt.subplots(figsize=(5.2, 4.6))
ax.scatter(reel.duree_reelle_observee / 60, reel.duree_min / 60, s=14, color=C_AVEC, alpha=0.7)
lim = max(reel.duree_reelle_observee.max(), reel.duree_min.max()) / 60
ax.plot([0, lim], [0, lim], color=C_GRIS, ls="--", lw=1, label="perfect match")
ax.set(title="Figure 14 — Evaluator: simulated vs observed duration", xlabel="Observed real duration (h)",
       ylabel="Simulated duration of the real order (h)"); ax.legend()
sauver(fig, "fig14_validation_evaluateur"); plt.show()

**Reading:** the evaluator reproduces the order of magnitude of real durations, with a median error of about 9 % per day and a correlation of 0.61. It tends to **underestimate** (points below the diagonal, total duration ≈ 89 % of real), especially for a few days when the courier had long unpredictable interruptions. It is **harsher than reality on lateness**: real couriers know how to speed up when a deadline approaches, which the model cannot. Simulated lateness must therefore be read **only as a comparison between strategies**, all evaluated in the same way, and not as absolute values.

### F.7 Results: KPI comparison

In [ ]:
# F.7 — KPI table and statistical tests

NOMS = {"reel": "Real (courier)", "sans_ml": "Optimised without ML", "avec_ml": "Optimised with ML"}
kpi = sim.groupby("strategie")[["distance_km", "deplacement_min", "duree_min", "attente_min", "n_retard", "retard_min"]].mean()
kpi = kpi.loc[["reel", "sans_ml", "avec_ml"]]
kpi_tot = sim.groupby("strategie")[["distance_km", "deplacement_min", "n_retard", "retard_min"]].sum().loc[["reel", "sans_ml", "avec_ml"]]
display(kpi.rename(index=NOMS).round(2))

def variation(a, b):
    return 100 * (b - a) / a

piv = {c: sim.pivot(index="journee", columns="strategie", values=c) for c in ("distance_km", "deplacement_min", "duree_min", "n_retard")}
gains = {
    "distance_avec_vs_reel_pct": variation(kpi.loc["reel", "distance_km"], kpi.loc["avec_ml", "distance_km"]),
    "distance_sans_vs_reel_pct": variation(kpi.loc["reel", "distance_km"], kpi.loc["sans_ml", "distance_km"]),
    "distance_avec_vs_sans_pct": variation(kpi.loc["sans_ml", "distance_km"], kpi.loc["avec_ml", "distance_km"]),
    "deplacement_avec_vs_reel_pct": variation(kpi.loc["reel", "deplacement_min"], kpi.loc["avec_ml", "deplacement_min"]),
    "deplacement_sans_vs_reel_pct": variation(kpi.loc["reel", "deplacement_min"], kpi.loc["sans_ml", "deplacement_min"]),
    "deplacement_avec_vs_sans_pct": variation(kpi.loc["sans_ml", "deplacement_min"], kpi.loc["avec_ml", "deplacement_min"]),
    "duree_avec_vs_reel_pct": variation(kpi.loc["reel", "duree_min"], kpi.loc["avec_ml", "duree_min"]),
    "duree_sans_vs_reel_pct": variation(kpi.loc["reel", "duree_min"], kpi.loc["sans_ml", "duree_min"]),
    "part_journees_ml_meilleur_que_sans_deplacement": float((piv["deplacement_min"].avec_ml < piv["deplacement_min"].sans_ml - 1e-9).mean()),
    "part_journees_ml_meilleur_que_reel_deplacement": float((piv["deplacement_min"].avec_ml < piv["deplacement_min"].reel - 1e-9).mean()),
    "p_wilcoxon_deplacement_avec_vs_sans": float(wilcoxon(piv["deplacement_min"].avec_ml, piv["deplacement_min"].sans_ml).pvalue),
    "p_wilcoxon_distance_avec_vs_sans": float(wilcoxon(piv["distance_km"].avec_ml, piv["distance_km"].sans_ml).pvalue),
    "p_wilcoxon_deplacement_avec_vs_reel": float(wilcoxon(piv["deplacement_min"].avec_ml, piv["deplacement_min"].reel).pvalue),
}
for k, v in gains.items():
    print(f"{k:<52} {v:10.4f}")

### F.8 Before / after charts

In [ ]:
# F.8 — KPI comparison (main figure)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))
for ax, (col, titre, unite) in zip(axes, [("distance_km", "Distance travelled", "km / day"),
                                         ("deplacement_min", "Travel time", "min / day"),
                                         ("n_retard", "Late pickups", "visits / day")]):
    vals = kpi[col].values
    ax.bar(list(NOMS.values()), vals, color=[C_REEL, C_SANS, C_AVEC], width=0.62)
    for i, v in enumerate(vals):
        ax.text(i, v, f"{v:.2f}" if col == "n_retard" else f"{v:.1f}", ha="center", va="bottom", fontsize=9)
    ax.set_title(titre); ax.set_ylabel(unite); ax.tick_params(axis="x", labelrotation=15, labelsize=8)
fig.suptitle(f"Figure 15 — Mean KPIs per day ({len(instances)} test days, λ = {LAMBDA:g})", fontweight="bold")
fig.tight_layout(); sauver(fig, "fig15_kpi_avant_apres"); plt.show()

# Distribution of gains per day
gain_j = 100 * (piv["deplacement_min"].reel - piv["deplacement_min"].avec_ml) / piv["deplacement_min"].reel
gain_s = 100 * (piv["deplacement_min"].reel - piv["deplacement_min"].sans_ml) / piv["deplacement_min"].reel
fig, ax = plt.subplots(figsize=(8, 3.6))
bins = np.linspace(min(gain_j.min(), gain_s.min()), max(gain_j.max(), gain_s.max()), 30)
ax.hist(gain_s, bins=bins, color=C_SANS, histtype="step", lw=2, label="Optimised without ML")
ax.hist(gain_j, bins=bins, color=C_AVEC, histtype="step", lw=2, label="Optimised with ML")
ax.axvline(0, color=C_GRIS, lw=1)
ax.set(title="Figure 16 — Travel-time gain per day compared with the real courier",
       xlabel="Gain (%) — positive = better than real", ylabel="Days"); ax.legend()
sauver(fig, "fig16_gains_par_journee"); plt.show()
print(f"Median gain per day: without ML {gain_s.median():.1f} %; with ML {gain_j.median():.1f} %")

**Reading the results (λ = 5, 150 test days):**
1. **Optimisation brings a clear gain over the real courier**, with or without ML: about **15 to 21 % less distance** and **6 to 7 % less travel time**. On more than 80 % of the days, the route optimised with ML is shorter in time than the real route (Wilcoxon test: p < 0.001).
2. **Without ML, this gain is paid for in lateness**: in simulation, the optimiser without ML has *more* late pickups than the real order replayed under the same conditions. Its formula underestimates times: on the **real** trips of the test set, its bias is about −5 min per trip (part E): it believes it can make a detour and arrive on time, then arrives late.
3. **With ML, the optimiser keeps most of the gain and reduces lateness**: about one third fewer late pickups than without ML, and fewer than the real order replayed in simulation. In return, it travels a little more distance than the optimiser without ML (about +8 %) for an almost identical travel time (about +1 %).

**Interim conclusion:** in this dataset, the added value of ML is **not** to shorten routes further; it is to **make the plan reliable**, i.e. to meet deadlines while keeping the gain from optimisation. It is a measured **trade-off**, not a free gain.

### F.9 A concrete example of a decision
Let us take a simulated day and look at the routes on the map: the grey line is the real order, the blue line is the order decided with ML.

In [ ]:
# F.9 — Example route before / after

ecarts_j = piv["distance_km"].reel - piv["distance_km"].avec_ml
k_ex = int(ecarts_j.sort_values().index[int(len(ecarts_j) * 0.75)])     # an "upper-typical" day, not the best one
inst = instances[k_ex]; v = inst["v"]
# local coordinates in km (more readable than degrees)
xk = (v.lng.values - v.lng.values[0]) * 111.32 * np.cos(np.radians(v.lat.values[0]))
yk = (v.lat.values - v.lat.values[0]) * 110.57
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), sharex=True, sharey=True)
for ax, s, coul in zip(axes, ["reel", "avec_ml"], [C_REEL, C_AVEC]):
    o = inst["resultats"][s]["ordre"]
    ax.plot(xk[o], yk[o], "-", color=coul, lw=1.6, zorder=2)
    ax.scatter(xk, yk, s=150, color="white", edgecolor=coul, lw=1.4, zorder=3)
    for rang, j in enumerate(o):
        ax.text(xk[j], yk[j], str(rang + 1), fontsize=7, ha="center", va="center", zorder=4)
    r = inst["resultats"][s]
    ax.set_title(f"{NOMS[s]} : {r['distance_km']:.1f} km, {r['deplacement_min']:.0f} min, {r['n_retard']} late", fontsize=10)
    ax.set_xlabel("East–west (km from the 1st visit)"); ax.set_aspect("equal")
axes[0].set_ylabel("North–south (km)")
fig.suptitle(f"Figure 17 — Day of courier {inst['courier_id']} ({str(inst['ds']).zfill(4)[:2]}/{str(inst['ds']).zfill(4)[2:]}), {inst['n']} visits",
             fontweight="bold")
fig.tight_layout(); sauver(fig, "fig17_exemple_tournee"); plt.show()

# One isolated decision: the first stop where the two orders diverge
o_r, o_m = inst["resultats"]["reel"]["ordre"], inst["resultats"]["avec_ml"]["ordre"]
pos = next((i for i in range(len(o_r)) if o_r[i] != o_m[i]), None)
if pos is not None:
    a = o_m[pos - 1]
    print(f"Decision #{pos}: the courier has just finished visit {a + 1}.")
    print(f"  - Real     : went to visit {o_r[pos] + 1} (distance {inst['dist'][a][o_r[pos]]:.2f} km)")
    print(f"  - With ML  : the optimiser picks visit {o_m[pos] + 1} (distance {inst['dist'][a][o_m[pos]]:.2f} km)")
    exemple_decision = {"rang": pos, "depuis": a + 1, "reel": o_r[pos] + 1, "ml": o_m[pos] + 1,
                        "dist_reel": inst["dist"][a][o_r[pos]], "dist_ml": inst["dist"][a][o_m[pos]]}

### F.10 Sensitivity analysis: the time / lateness trade-off (λ)
λ sets the importance given to lateness. We rerun the simulation for several values of λ on 50 days.

In [ ]:
# F.10 — Sensitivity to λ

LAMBDAS = [0.0, 1.0, 5.0, 20.0]
sens = []
for lam in LAMBDAS:
    for s in ("sans_ml", "avec_ml"):
        rs = [simuler(inst, s, lam=lam) for inst in instances[:50]]
        sens.append({"lambda": lam, "strategie": s, "deplacement_min": np.mean([r["deplacement_min"] for r in rs]),
                     "n_retard": np.mean([r["n_retard"] for r in rs]), "retard_min": np.mean([r["retard_min"] for r in rs])})
reel50 = sim[(sim.strategie == "reel") & (sim.journee < 50)][["deplacement_min", "n_retard"]].mean()
sens = pd.DataFrame(sens)
display(sens.round(3))
fig, ax = plt.subplots(figsize=(7.5, 4))
for s, coul in (("sans_ml", C_SANS), ("avec_ml", C_AVEC)):
    d = sens[sens.strategie == s]
    ax.plot(d.deplacement_min, d.n_retard, "o-", color=coul, lw=2, label=NOMS[s])
    for i, (_, r) in enumerate(d.iterrows()):
        ax.annotate(f"λ={r['lambda']:g}", (r.deplacement_min, r.n_retard), textcoords="offset points",
                    xytext=(6, 6) if i % 2 == 0 else (6, -12), fontsize=8, color=C_GRIS)
ax.scatter([reel50.deplacement_min], [reel50.n_retard], color=C_REEL, s=70, zorder=3, label="Real (courier)")
ax.set(title="Figure 18 — Travel time / lateness trade-off by λ (50 days)",
       xlabel="Mean travel time (min / day)", ylabel="Late pickups (mean / day)"); ax.legend()
sauver(fig, "fig18_sensibilite_lambda"); plt.show()

**Reading:**
* With **λ = 0** (lateness ignored), ML brings no advantage (it even does slightly worse): both optimisers only try to shorten the route.
* As soon as lateness counts (**λ ≥ 1**), the optimiser **with ML** systematically has fewer late pickups than the optimiser **without ML**.
* Above all, **increasing λ is not enough to fix the optimiser without ML**: its lateness plateaus, because the problem comes from its **time estimates**, not from the penalty. A good penalty does not compensate for a bad prediction: this is exactly the role of ML in this project.

### F.11 Heuristic quality: comparison with the exact optimum (MILP)
We solve model F.1 **exactly** with HiGHS on sub-problems of 8 visits (ML matrix of the departure hour) and compare with the heuristic's result.

In [ ]:
# F.11 — Exact MILP (HiGHS) vs heuristic

def milp_tsptw(T, ech, tau, lam, limite_s=60):
    """T: (n+1)x(n+1) matrix, node 0 = starting position. Returns (optimal cost, order, computation time)."""
    n = T.shape[0] - 1
    fin = n + 1                                              # dummy end node
    arcs = [(i, j) for i in range(n + 1) for j in range(1, n + 2) if i != j and not (i == 0 and j == fin)]
    na = len(arcs); ia = {a: k for k, a in enumerate(arcs)}
    nv = na + n + n                                          # x_arcs, a_1..a_n, L_1..L_n
    A_ = lambda j: na + (j - 1); L_ = lambda j: na + n + (j - 1)
    c = np.zeros(nv)
    for (i, j), k in ia.items():
        c[k] = T[i, j] if j != fin else 0.0
    c[na + n:] = lam
    rows, lo, hi = [], [], []
    def ajout(coefs, l, h):
        r = np.zeros(nv)
        for k, v in coefs: r[k] += v
        rows.append(r); lo.append(l); hi.append(h)
    for j in range(1, n + 2):                                # one entry per node (and towards the end)
        ajout([(ia[(i, j)], 1) for i in range(n + 1) if (i, j) in ia], 1, 1)
    for i in range(n + 1):                                   # one exit per node
        ajout([(ia[(i, j)], 1) for j in range(1, n + 2) if (i, j) in ia], 1, 1)
    Mbig = tau + T.sum() + 1000
    for (i, j), k in ia.items():                             # time propagation
        if j == fin:
            continue
        if i == 0:   # a_j >= tau + T0j - M(1-x)  ->  a_j - M x >= tau + T0j - M
            ajout([(A_(j), 1), (k, -Mbig)], tau + T[0, j] - Mbig, np.inf)
        else:        # a_j - a_i - M x >= T_ij - M
            ajout([(A_(j), 1), (A_(i), -1), (k, -Mbig)], T[i, j] - Mbig, np.inf)
    for j in range(1, n + 1):                                # L_j >= a_j - l_j
        ajout([(L_(j), 1), (A_(j), -1)], -ech[j], np.inf)
    integ = np.zeros(nv); integ[:na] = 1
    lb = np.zeros(nv); ub = np.full(nv, np.inf); ub[:na] = 1
    t0 = time.time()
    res = milp(c, constraints=LinearConstraint(np.array(rows), lo, hi), integrality=integ,
               bounds=Bounds(lb, ub), options={"time_limit": limite_s})
    duree = time.time() - t0
    x = res.x[:na] > 0.5
    succ = {i: j for (i, j), k in ia.items() if x[k]}
    ordre, cur = [], 0
    while succ[cur] != fin:
        cur = succ[cur]; ordre.append(cur)
    return res.fun, ordre, duree

comparaison_milp = []
for inst in instances[:30]:
    noeuds = list(range(9))                                  # start + 8 visits
    h = idx_heure(inst["fin"][0])
    T = np.array(inst["M_ml"][h])[np.ix_(noeuds, noeuds)]
    ech = np.array(inst["echeance"])[noeuds]                 # real deadlines of these visits
    tau = float(inst["fin"][0])                              # same departure hour as reality
    opt, ordre_opt, duree = milp_tsptw(T, ech, tau, LAMBDA)
    Mfix = [T.tolist()] * len(HEURES)                         # same matrix at every hour (fair comparison)
    t1 = time.time(); route, ch = planifier(0, tau, list(range(1, 9)), Mfix, ech, LAMBDA); dh = time.time() - t1
    comparaison_milp.append({"journee": inst["ds"], "cout_milp": opt, "cout_heuristique": ch,
                             "ecart_pct": 100 * (ch - opt) / opt, "temps_milp_s": duree, "temps_heuristique_s": dh})
cmp_milp = pd.DataFrame(comparaison_milp)
display(cmp_milp.describe().loc[["mean", "50%", "max"]].round(4))
print(f"The heuristic finds the proven optimum in {(cmp_milp.ecart_pct < 1e-6).mean():.0%} of the 30 instances.")

**Why not use the MILP everywhere?** Exact computation time explodes with the number of visits (NP-hard problem), whereas an instant answer is needed at every stop. The MILP serves as a **quality certificate**: it proves that the heuristic is very close to the optimum on instances where the optimum can be computed.

### F.12 Optimistic scenario: if all tasks were known from the morning
As a (non-realistic) **bound**, we rerun with all tasks known at the start. This measures what we lose because of the progressive arrival of orders.

In [ ]:
# F.12 — "Perfect information" bound

borne = []
for inst in instances[:50]:
    r = simuler(inst, "avec_ml", toutes_connues=True)
    borne.append({"deplacement_min": r["deplacement_min"], "distance_km": r["distance_km"], "n_retard": r["n_retard"]})
borne = pd.DataFrame(borne).mean()
ref50 = sim[sim.journee < 50].groupby("strategie")[["deplacement_min", "distance_km", "n_retard"]].mean()
display(pd.concat([ref50.rename(index=NOMS), borne.rename("With ML — tasks known at start (bound)").to_frame().T]).round(2))

## G. Exports for the report and the web platform

In [ ]:

# G — Exports JSON / CSV
def to_py(o):
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, np.ndarray): return o.tolist()
    raise TypeError(type(o))

resultats_optim = {
    "lambda": LAMBDA, "n_journees": len(instances), "heures_matrices": [int(h) for h in HEURES],
    "kpi_moyens": kpi.rename(index=NOMS).round(4).reset_index(names="strategie").to_dict(orient="records"),
    "kpi_totaux": kpi_tot.rename(index=NOMS).round(3).reset_index(names="strategie").to_dict(orient="records"),
    "gains": gains,
    "gain_median_deplacement_pct": {"sans_ml": float(gain_s.median()), "avec_ml": float(gain_j.median())},
    "gains_par_journee_pct": {"sans_ml": gain_s.round(3).tolist(), "avec_ml": gain_j.round(3).tolist()},
    "validation_evaluateur": {"ratio_duree": ratio, "correlation": corr_duree, "erreur_relative_mediane": err_duree},
    "sensibilite_lambda": sens.round(4).to_dict(orient="records"),
    "reel_50": reel50.round(4).to_dict(),
    "milp_vs_heuristique": {"ecart_moyen_pct": float(cmp_milp.ecart_pct.mean()), "ecart_max_pct": float(cmp_milp.ecart_pct.max()),
                            "part_optimum": float((cmp_milp.ecart_pct < 1e-6).mean()),
                            "temps_milp_moyen_s": float(cmp_milp.temps_milp_s.mean()),
                            "temps_heuristique_moyen_s": float(cmp_milp.temps_heuristique_s.mean()), "n_instances": len(cmp_milp)},
    "borne_information_parfaite": borne.round(4).to_dict(),
    "retards_observes_moyens": float(np.mean(retards_obs)),
    "exemple_decision": exemple_decision,
    "exemple": {"journee": k_ex, "courier_id": instances[k_ex]["courier_id"], "ds": instances[k_ex]["ds"]},
    "par_journee": sim.round(3).to_dict(orient="records"),
}
(EXP_DIR / "resultats_ml.json").write_text(json.dumps(resultats_ml, default=to_py, ensure_ascii=False, indent=1), encoding="utf-8")
(EXP_DIR / "resultats_optimisation.json").write_text(json.dumps(resultats_optim, default=to_py, ensure_ascii=False, indent=1), encoding="utf-8")
(EXP_DIR / "donnees_dashboard.json").write_text(json.dumps(eda, default=to_py, ensure_ascii=False), encoding="utf-8")
sim.to_csv(EXP_DIR / "resultats_par_journee.csv", index=False)
tableau_ml.round(4).to_csv(EXP_DIR / "comparaison_modeles.csv")

# --- Scenarios for the web simulator: 24 days (≤ 20 visits), same matrices as the simulation ---
web_sc = []
for k, inst in enumerate(instances):
    if inst["n"] > 20 or len(web_sc) >= 24:
        continue
    v = inst["v"]
    r1 = lambda M: M
    web_sc.append({"id": k, "courier_id": inst["courier_id"], "ds": inst["ds"], "n": inst["n"],
                   "lng": v.lng.round(5).tolist(), "lat": v.lat.round(5).tolist(), "n_colis": v.n_colis.tolist(),
                   "aoi_type": v.aoi_type.tolist(),
                   "connue": np.round(inst["connue"], 1).tolist(), "echeance": np.round(inst["echeance"], 1).tolist(),
                   "debut": np.round(inst["debut"], 1).tolist(), "fin": np.round(inst["fin"], 1).tolist(),
                   "dist": inst["dist"], "resultats_notebook": {s: {c: r[c] for c in r if c != "ordre"} for s, r in inst["resultats"].items()},
                   "M_sans": r1(inst["M_sans"]), "M_ml": r1(inst["M_ml"]), "M_eval": r1(inst["M_eval"])})
(WEB_DIR / "scenarios.json").write_text(json.dumps({"heure_min": H0, "heure_max": H1, "lambda_defaut": LAMBDA,
                                                    "scenarios": web_sc}, default=to_py))
# Sample of 2,000 parcels for the data explorer
raw.sample(2000, random_state=SEED).sort_values(["courier_id", "ds"]).to_csv(WEB_DIR / "echantillon_colis.csv", index=False)
for f in ("resultats_ml.json", "resultats_optimisation.json", "donnees_dashboard.json"):
    (WEB_DIR / f).write_text((EXP_DIR / f).read_text(encoding="utf-8"), encoding="utf-8")
# Single file loaded by the website (also works when opening index.html by double-click)
ech_web = raw.sample(2000, random_state=SEED).sort_values(["courier_id", "ds"])
paquet = {"ml": resultats_ml, "optim": {k: v for k, v in resultats_optim.items() if k != "par_journee"},
          "eda": eda, "scenarios": json.loads((WEB_DIR / "scenarios.json").read_text()),
          "echantillon": {"colonnes": list(ech_web.columns), "lignes": ech_web.astype(object).where(ech_web.notna(), None).values.tolist()}}
(WEB_DIR / "donnees.js").write_text("window.PROJET = " + json.dumps(paquet, default=to_py, ensure_ascii=False) + ";", encoding="utf-8")
print("Exports done:", sorted(p.name for p in EXP_DIR.iterdir()), "|", len(web_sc), "web scenarios")

## H. Conclusion

*The figures below are recomputed from the results above (they are not typed by hand).*

In [ ]:
# H — Automatic numerical summary

k_r, k_s, k_m = kpi.loc["reel"], kpi.loc["sans_ml"], kpi.loc["avec_ml"]
b = tableau_ml.loc["Baseline (historical averages)"]; f = tableau_ml.loc[NOM_FINAL]
print(f"""1. Problem: dynamically order a courier's pickups, subject to deadlines.
2. EDA: windows of {duree_fen.median():.0f} min; {eda['indicateurs']['taux_retard_reel']:.2%} real lateness;
   only {eda['indicateurs']['part_connue_avant_debut']:.0%} of visits known before the first pickup -> dynamic optimisation.
3. ML: {NOM_FINAL} — MAE {f.MAE:.2f} min versus {b.MAE:.2f} min for the historical formula ({100*(f.MAE-b.MAE)/b.MAE:+.1f} %), R² {f.R2:.2f}.
4. Optimisation ({len(instances)} test days, λ = {LAMBDA:g}):
   distance       : real {k_r.distance_km:.2f} km | without ML {k_s.distance_km:.2f} | with ML {k_m.distance_km:.2f} km/day
   travel time    : real {k_r.deplacement_min:.1f} min | without ML {k_s.deplacement_min:.1f} | with ML {k_m.deplacement_min:.1f} min/day
   lateness       : real {k_r.n_retard:.2f} | without ML {k_s.n_retard:.2f} | with ML {k_m.n_retard:.2f} visits/day
5. Value of optimisation (with ML vs real): distance {gains['distance_avec_vs_reel_pct']:+.1f} %, travel time {gains['deplacement_avec_vs_reel_pct']:+.1f} %.
6. Added value of ML (with vs without ML): time {gains['deplacement_avec_vs_sans_pct']:+.1f} % (p = {gains['p_wilcoxon_deplacement_avec_vs_sans']:.2g}),
   distance {gains['distance_avec_vs_sans_pct']:+.1f} % (p = {gains['p_wilcoxon_distance_avec_vs_sans']:.2g}).
7. Heuristic: mean gap of {cmp_milp.ecart_pct.mean():.2f} % to the MILP optimum on 30 instances of 8 visits.""")

### Summary
* **Initial problem:** order, at each stop, the pickups of a courier whose tasks arrive during the day, under 2-hour deadlines.
* **EDA:** couriers already meet deadlines very well; fewer than half of the tasks are known in the morning, hence a **dynamic optimisation**; time depends on distance but also on hour, place and unobserved elements.
* **ML:** Gradient Boosting predicts **unbiased** times, unlike the historical formula which underestimates them; its accuracy remains limited (R² ≈ 0.1).
* **Optimised decisions:** a TSPTW-type MILP, solved at each stop by a 2-opt/or-opt heuristic validated against the exact optimum (HiGHS).
* **Simulated gains:** optimisation shortens routes compared with the field; ML makes it possible to keep this gain **without degrading** deadline compliance.

### Limitations
* **Time gains are simulated**, not observed: a route that was never driven can only be evaluated with a model. We mitigated this with an **independent** evaluator (other couriers) validated on real routes (Figure 14). The **distance**, however, is computed exactly.
* Real couriers know things absent from the data (closed gate, absent customer, phone call, deliveries to make in parallel): part of the "gap" with reality cannot be recovered in practice.
* Distance is straight-line: the road network (provided in LaDe, `roads.csv`) would give more faithful distances.
* The year is unknown: the day of the week could not be used.

### Improvement ideas
* Road distances from `roads.csv`.
* Multi-courier version (VRPTW): reassign visits between neighbouring couriers.
* Predict the **arrival of future tasks** to anticipate detours.
* Time model with confidence intervals (quantile regression) for robust routes.